# Code Pattern Analysis — Significant Confused Pairs

**Pipeline:**
1. Load `confusion_matrix_cwe.csv` and select **significant** pairs directly from it
   - Both directions `(A→B)` and `(B→A)` merged into one undirected pair
   - Pair kept only if **combined count ≥ threshold AND max confusion rate ≥ threshold**
2. Extract regex-based code features in memory-safe chunks
3. Compare feature frequencies between each significant pair
4. Export per-pair comparison tables + master CSV

## 1. Imports & Config

In [1]:
import pandas as pd
import numpy as np
import re
import gc
from itertools import combinations

# ─── CONFIG ───────────────────────────────────────────────────────────────────
CM_FILE        = "confusion_matrix_cwe.csv"   # rows = true class, cols = predicted class
DATASET_FILE   = "juliet_cwe_dataset.csv"
CODE_COLUMN    = "code_clean"
CWE_COLUMN     = "cwe"

# Significance thresholds — a merged pair (A,B) is kept when:
#   combined_count  = cm[A→B] + cm[B→A]  >=  MIN_COMBINED_COUNT
#   AND
#   max_rate        = max(cm[A→B]/total_A, cm[B→A]/total_B)  >=  MIN_CONFUSION_RATE
MIN_COMBINED_COUNT  = 4      # at least 4 cross-misclassifications total
MIN_CONFUSION_RATE  = 0.005  # at least 0.5 % of either class misclassified as the other

CHUNK_SIZE = 5_000           # rows per chunk for feature extraction
# ──────────────────────────────────────────────────────────────────────────────
print("✅ Config ready")

✅ Config ready


## 2. Load Confusion Matrix & Select Significant Pairs

Both directions `A→B` and `B→A` are merged:
- `combined_count = cm[A,B] + cm[B,A]`
- `rate_A = cm[A,B] / row_total_A`  (fraction of A samples called B)
- `rate_B = cm[B,A] / row_total_B`  (fraction of B samples called A)
- `max_rate = max(rate_A, rate_B)`

A pair survives when **both** `combined_count >= MIN_COMBINED_COUNT` **and** `max_rate >= MIN_CONFUSION_RATE`.

In [2]:
cm = pd.read_csv(CM_FILE, index_col=0)

# Align: keep only CWEs that appear in both index and columns
shared = cm.index.intersection(cm.columns)
cm = cm.loc[shared, shared]

row_totals = cm.sum(axis=1)   # total samples per true class

sig_pairs = []

cwes = list(cm.index)
for i, cwe_a in enumerate(cwes):
    for cwe_b in cwes[i+1:]:          # upper triangle only → each pair once

        count_ab = cm.loc[cwe_a, cwe_b]   # A misclassified as B
        count_ba = cm.loc[cwe_b, cwe_a]   # B misclassified as A
        combined = count_ab + count_ba

        if combined < MIN_COMBINED_COUNT:
            continue

        rate_a = count_ab / row_totals[cwe_a] if row_totals[cwe_a] > 0 else 0
        rate_b = count_ba / row_totals[cwe_b] if row_totals[cwe_b] > 0 else 0
        max_rate = max(rate_a, rate_b)

        if max_rate < MIN_CONFUSION_RATE:
            continue

        sig_pairs.append({
            "CWE_A":          cwe_a,
            "CWE_B":          cwe_b,
            "Count_A_to_B":   int(count_ab),
            "Count_B_to_A":   int(count_ba),
            "Combined_Count": int(combined),
            "Rate_A_to_B":    round(float(rate_a), 6),
            "Rate_B_to_A":    round(float(rate_b), 6),
            "Max_Rate":       round(float(max_rate), 6),
        })

pairs_df = (
    pd.DataFrame(sig_pairs)
    .sort_values("Combined_Count", ascending=False)
    .reset_index(drop=True)
)

confused_cwes = set(pairs_df["CWE_A"]) | set(pairs_df["CWE_B"])

print(f"Significant pairs  : {len(pairs_df)}")
print(f"Unique CWEs        : {len(confused_cwes)}")
print(f"Thresholds         : count ≥ {MIN_COMBINED_COUNT}, rate ≥ {MIN_CONFUSION_RATE}")
print()
print(pairs_df.to_string(index=False))

Significant pairs  : 3
Unique CWEs        : 5
Thresholds         : count ≥ 4, rate ≥ 0.005

 CWE_A  CWE_B  Count_A_to_B  Count_B_to_A  Combined_Count  Rate_A_to_B  Rate_B_to_A  Max_Rate
CWE195 CWE197            10             0              10     0.013514          0.0  0.013514
CWE194 CWE197             8             0               8     0.010811          0.0  0.010811
CWE114 CWE134             4             0               4     0.009009          0.0  0.009009


### Threshold Sensitivity Check
Run this cell to see how many pairs survive at different threshold combinations — helps you tune `MIN_COMBINED_COUNT` and `MIN_CONFUSION_RATE` without re-running the full pipeline.

In [3]:
count_thresholds = [2, 4, 6, 10]
rate_thresholds  = [0.001, 0.005, 0.01, 0.02]

# Pre-compute all raw pairs (no thresholding)
_all_raw = []
for i, cwe_a in enumerate(cwes):
    for cwe_b in cwes[i+1:]:
        cnt_ab = cm.loc[cwe_a, cwe_b]
        cnt_ba = cm.loc[cwe_b, cwe_a]
        comb   = cnt_ab + cnt_ba
        ra = cnt_ab / row_totals[cwe_a] if row_totals[cwe_a] > 0 else 0
        rb = cnt_ba / row_totals[cwe_b] if row_totals[cwe_b] > 0 else 0
        _all_raw.append((comb, max(ra, rb)))

rows = []
for ct in count_thresholds:
    for rt in rate_thresholds:
        n = sum(1 for (c, r) in _all_raw if c >= ct and r >= rt)
        rows.append({"min_count": ct, "min_rate": rt, "pairs_kept": n})

sens = pd.DataFrame(rows).pivot(index="min_count", columns="min_rate", values="pairs_kept")
print("Pairs kept at each (min_count × min_rate) combination:")
print(sens.to_string())
print("\n★  Currently using:", MIN_COMBINED_COUNT, "count /", MIN_CONFUSION_RATE, "rate")

Pairs kept at each (min_count × min_rate) combination:
min_rate   0.001  0.005  0.010  0.020
min_count                            
2             12      3      2      0
4              5      3      2      0
6              4      2      2      0
10             1      1      1      0

★  Currently using: 4 count / 0.005 rate


## 3. Feature Library — compiled once

In [4]:
_RAW_FEATURES = [
    # memory
    ("uses_malloc",           r'\bmalloc\s*\(',                        "memory"),
    ("uses_calloc",           r'\bcalloc\s*\(',                        "memory"),
    ("uses_realloc",          r'\brealloc\s*\(',                       "memory"),
    ("uses_free",             r'\bfree\s*\(',                          "memory"),
    ("uses_new",              r'\bnew\b',                              "memory"),
    ("uses_delete",           r'\bdelete\b',                           "memory"),
    ("pointer_arithmetic",    r'\w+\s*[+\-]\s*\d+\s*(?=\[|->|\))',    "memory"),
    ("array_index_variable",  r'\w+\s*\[\s*[a-zA-Z_]\w*\s*\]',        "memory"),
    ("stack_buffer",          r'\bchar\s+\w+\s*\[\s*\d+\s*\]',         "memory"),
    ("memcpy_present",        r'\bmemcpy\s*\(',                        "memory"),
    ("memmove_present",       r'\bmemmove\s*\(',                       "memory"),
    ("memset_present",        r'\bmemset\s*\(',                        "memory"),
    ("sizeof_used",           r'\bsizeof\s*\(',                        "memory"),
    # integer
    ("cast_to_int",           r'\(\s*int\s*\)',                        "integer"),
    ("cast_to_short",         r'\(\s*short\s*\)',                      "integer"),
    ("cast_to_char",          r'\(\s*char\s*\)',                       "integer"),
    ("cast_to_unsigned",      r'\(\s*unsigned\b[^)]*\)',               "integer"),
    ("cast_to_long",          r'\(\s*long\b[^)]*\)',                   "integer"),
    ("unsigned_type",         r'\bunsigned\s+(int|long|short|char)\b', "integer"),
    ("signed_type",           r'\bsigned\s+(int|long|short|char)\b',   "integer"),
    ("integer_literal_hex",   r'0[xX][0-9a-fA-F]+',                   "integer"),
    ("arithmetic_shift",      r'<<|>>',                                "integer"),
    ("INT_MAX_MIN_used",       r'\b(INT_MAX|INT_MIN|UINT_MAX|LONG_MAX|SHRT_MAX)\b', "integer"),
    # string
    ("strcpy_unsafe",         r'\bstrcpy\s*\(',                        "string_ops"),
    ("strncpy_safe",          r'\bstrncpy\s*\(',                       "string_ops"),
    ("strcat_unsafe",         r'\bstrcat\s*\(',                        "string_ops"),
    ("strncat_safe",          r'\bstrncat\s*\(',                       "string_ops"),
    ("strlen_present",        r'\bstrlen\s*\(',                        "string_ops"),
    ("strcmp_present",        r'\bstrcmp\s*\(',                        "string_ops"),
    ("gets_unsafe",           r'\bgets\s*\(',                          "string_ops"),
    ("fgets_safe",            r'\bfgets\s*\(',                         "string_ops"),
    ("sprintf_unsafe",        r'\bsprintf\s*\(',                       "string_ops"),
    ("snprintf_safe",         r'\bsnprintf\s*\(',                      "string_ops"),
    # format string
    ("printf_present",        r'\bprintf\s*\(',                        "format_string"),
    ("fprintf_present",       r'\bfprintf\s*\(',                       "format_string"),
    ("printf_no_fmt_literal", r'\bprintf\s*\(\s*\w+\s*[,)]',          "format_string"),
    ("format_spec_s",         r'%s',                                   "format_string"),
    ("format_spec_d",         r'%d|%i',                                "format_string"),
    ("format_spec_n",         r'%n',                                   "format_string"),
    ("vprintf_family",        r'\bv(printf|fprintf|sprintf|snprintf)\s*\(', "format_string"),
    # control flow
    ("has_for_loop",          r'\bfor\s*\(',                           "control_flow"),
    ("has_while_loop",        r'\bwhile\s*\(',                         "control_flow"),
    ("has_do_while",          r'\bdo\s*\{',                            "control_flow"),
    ("early_return",          r'\breturn\b',                           "control_flow"),
    ("goto_present",          r'\bgoto\b',                             "control_flow"),
    ("switch_present",        r'\bswitch\s*\(',                        "control_flow"),
    # error handling
    ("null_check_present",    r'(==|!=)\s*(NULL|nullptr)',             "error_handling"),
    ("null_assign",           r'=\s*(NULL|nullptr)\s*;',               "error_handling"),
    ("errno_used",            r'\berrno\b',                            "error_handling"),
    ("assert_used",           r'\bassert\s*\(',                        "error_handling"),
    ("try_catch",             r'\btry\s*\{|\bcatch\s*\(',              "error_handling"),
    # resource management
    ("fopen_present",         r'\bfopen\s*\(',                         "resource_mgmt"),
    ("fclose_present",        r'\bfclose\s*\(',                        "resource_mgmt"),
    ("socket_present",        r'\bsocket\s*\(',                        "resource_mgmt"),
    ("close_fd",              r'\bclose\s*\(',                         "resource_mgmt"),
    # dangerous apis
    ("system_call",           r'\bsystem\s*\(',                        "dangerous_api"),
    ("exec_family",           r'\b(execve|execl|execlp|execvp)\s*\(',  "dangerous_api"),
    ("rand_present",          r'\brand\s*\(',                          "dangerous_api"),
    ("getenv_present",        r'\bgetenv\s*\(',                        "dangerous_api"),
    ("atoi_atol",             r'\b(atoi|atol|atof)\s*\(',              "dangerous_api"),
]

FEATURES       = [(n, re.compile(p, re.DOTALL), g) for n, p, g in _RAW_FEATURES]
FEATURE_NAMES  = [f[0] for f in FEATURES]
FEATURE_GROUPS = {f[0]: f[2] for f in FEATURES}

print(f"✅ {len(FEATURES)} patterns compiled across "
      f"{len(set(f[2] for f in FEATURES))} groups")

✅ 60 patterns compiled across 8 groups


## 4. Chunked Feature Extraction
Only processes rows belonging to the significant CWEs.  
Accumulates `(hit_count, total_count)` per `(CWE, feature)` — discards each chunk immediately.

In [5]:
# {cwe -> {feature -> [hits, total]}}
accum = {cwe: {feat: [0, 0] for feat in FEATURE_NAMES} for cwe in confused_cwes}

chunk_num  = 0
total_kept = 0

for chunk in pd.read_csv(
    DATASET_FILE,
    usecols=[CWE_COLUMN, CODE_COLUMN],
    chunksize=CHUNK_SIZE,
    dtype={CWE_COLUMN: str, CODE_COLUMN: str},
    na_filter=False,
):
    chunk_num += 1
    chunk = chunk[chunk[CWE_COLUMN].isin(confused_cwes)]
    if chunk.empty:
        continue

    total_kept  += len(chunk)
    code_series  = chunk[CODE_COLUMN]
    cwe_series   = chunk[CWE_COLUMN]

    for feat_name, compiled_pat, _ in FEATURES:
        hits = code_series.str.contains(compiled_pat, regex=True).fillna(False)
        for cwe in cwe_series.unique():
            mask = cwe_series == cwe
            accum[cwe][feat_name][0] += int(hits[mask].sum())
            accum[cwe][feat_name][1] += int(mask.sum())

    del chunk, code_series, cwe_series, hits
    gc.collect()

    if chunk_num % 10 == 0:
        print(f"  chunk {chunk_num:>4}  |  kept so far: {total_kept:,}")

print(f"\n✅ Done — {chunk_num} chunks, {total_kept:,} rows kept")

/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_1639/1831457774.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True).fillna(False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_1639/1831457774.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True).fillna(False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_1639/1831457774.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True).fillna(False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_1639/1831457774.py:24: UserWarning: This pattern is interpreted as a regul

  chunk   20  |  kept so far: 13,449


/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_1639/1831457774.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True).fillna(False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_1639/1831457774.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True).fillna(False)



✅ Done — 36 chunks, 20,734 rows kept


## 5. Per-CWE Frequency Table

In [6]:
freq_rows = []
for cwe, feat_dict in accum.items():
    row = {CWE_COLUMN: cwe}
    for feat_name, (hits, total) in feat_dict.items():
        row[feat_name] = round(hits / total, 4) if total > 0 else 0.0
    freq_rows.append(row)

cwe_freq = (
    pd.DataFrame(freq_rows)
    .set_index(CWE_COLUMN)
    .sort_index()
)

print(f"Per-CWE frequency table: {cwe_freq.shape}  (rows=CWEs, cols=features)")
cwe_freq

Per-CWE frequency table: (5, 60)  (rows=CWEs, cols=features)


,uses_malloc,uses_calloc,uses_realloc,uses_free,uses_new,uses_delete,pointer_arithmetic,array_index_variable,stack_buffer,memcpy_present,...,try_catch,fopen_present,fclose_present,socket_present,close_fd,system_call,exec_family,rand_present,getenv_present,atoi_atol
cwe,,,,,,,,,,,,,,,,,,,,,
CWE114,0.0000,0.0,0.0,0.0000,0.0270,0.0270,0.3101,0.2067,0.3236,0.0000,...,0.0,0.1034,0.1034,0.2067,0.0,0.0,0.0,0.0,0.0,0.0000
CWE134,0.0000,0.0,0.0,0.0000,0.0262,0.0262,0.3698,0.2466,0.3519,0.0000,...,0.0,0.1233,0.1233,0.2466,0.0,0.0,0.0,0.0,0.0,0.0000
CWE194,0.1515,0.0,0.0,0.1515,0.0269,0.0269,0.6061,0.6178,0.4545,0.1515,...,0.0,0.0000,0.0000,0.2065,0.0,0.0,0.0,0.0,0.0,0.3098
CWE195,0.1515,0.0,0.0,0.1515,0.0269,0.0269,0.6061,0.6178,0.4545,0.1515,...,0.0,0.0000,0.0000,0.2065,0.0,0.0,0.0,0.0,0.0,0.3098
CWE197,0.0000,0.0,0.0,0.0000,0.0269,0.0269,0.0000,0.3094,0.0000,0.0000,...,0.0,0.0000,0.0000,0.2063,0.0,0.0,0.0,0.0,0.0,0.3094


## 6. Pairwise Comparison

For each significant merged pair `(A, B)`:
- `Δ = freq(A) − freq(B)` per feature
- **|Δ| ≥ 0.10** → discriminating signal the model isn't leveraging
- **|Δ| < 0.05** → ambiguous — both classes look identical here, likely confusion trigger

Direction labels in the report use `Count_A_to_B` / `Count_B_to_A` from the confusion matrix  
so you can see which direction is the dominant failure mode.

In [7]:
master_rows    = []
pair_summaries = {}
SEP = "─" * 76

for _, pr in pairs_df.iterrows():
    cwe_a = pr["CWE_A"]
    cwe_b = pr["CWE_B"]

    if cwe_a not in cwe_freq.index or cwe_b not in cwe_freq.index:
        print(f"  ⚠️  Skipping ({cwe_a}, {cwe_b}): not found in dataset")
        continue

    fa    = cwe_freq.loc[cwe_a]
    fb    = cwe_freq.loc[cwe_b]
    delta = (fa - fb).round(4)

    # Dominant direction label
    if pr["Count_A_to_B"] >= pr["Count_B_to_A"]:
        dom_dir = f"{cwe_a}→{cwe_b} ({pr['Count_A_to_B']} times)"
    else:
        dom_dir = f"{cwe_b}→{cwe_a} ({pr['Count_B_to_A']} times)"

    pair_df = pd.DataFrame({
        "Feature":         FEATURE_NAMES,
        "Group":           [FEATURE_GROUPS[f] for f in FEATURE_NAMES],
        f"Freq_{cwe_a}":   fa.values.round(4),
        f"Freq_{cwe_b}":   fb.values.round(4),
        "Delta_A_minus_B": delta.values,
        "Abs_Delta":       delta.abs().values,
    }).sort_values("Abs_Delta", ascending=False).reset_index(drop=True)

    pair_summaries[(cwe_a, cwe_b)] = {
        "combined_count": int(pr["Combined_Count"]),
        "count_ab":       int(pr["Count_A_to_B"]),
        "count_ba":       int(pr["Count_B_to_A"]),
        "rate_ab":        float(pr["Rate_A_to_B"]),
        "rate_ba":        float(pr["Rate_B_to_A"]),
        "max_rate":       float(pr["Max_Rate"]),
        "dominant_dir":   dom_dir,
        "discriminating": pair_df[pair_df["Abs_Delta"] >= 0.10].head(10),
        "ambiguous":      pair_df[pair_df["Abs_Delta"] <  0.05].head(10),
        "pair_df":        pair_df,
    }

    for _, r in pair_df.iterrows():
        master_rows.append({
            "CWE_A":            cwe_a,
            "CWE_B":            cwe_b,
            "Combined_Count":   int(pr["Combined_Count"]),
            "Count_A_to_B":     int(pr["Count_A_to_B"]),
            "Count_B_to_A":     int(pr["Count_B_to_A"]),
            "Max_Rate":         float(pr["Max_Rate"]),
            "Feature":          r["Feature"],
            "Group":            r["Group"],
            f"Freq_{cwe_a}":    r[f"Freq_{cwe_a}"],
            f"Freq_{cwe_b}":    r[f"Freq_{cwe_b}"],
            "Delta_A_minus_B":  r["Delta_A_minus_B"],
            "Abs_Delta":        r["Abs_Delta"],
        })

master_df = pd.DataFrame(master_rows)
print(f"✅ Master table: {len(master_df):,} rows  "
      f"({len(pair_summaries)} pairs × {len(FEATURE_NAMES)} features)")

✅ Master table: 180 rows  (3 pairs × 60 features)


## 7. Per-Pair Reports

In [8]:
for (cwe_a, cwe_b), info in pair_summaries.items():
    print(SEP)
    print(f"  Pair: {cwe_a}  ↔  {cwe_b}")
    print(f"  Combined misclassifications : {info['combined_count']}  "
          f"({cwe_a}→{cwe_b}: {info['count_ab']},  "
          f"{cwe_b}→{cwe_a}: {info['count_ba']})")
    print(f"  Error rates                 : {cwe_a}→{cwe_b}={info['rate_ab']*100:.2f}%,  "
          f"{cwe_b}→{cwe_a}={info['rate_ba']*100:.2f}%")
    print(f"  Dominant direction          : {info['dominant_dir']}")
    print(SEP)

    disc = info["discriminating"]
    if not disc.empty:
        print("  ▶ Discriminating features (|Δ| ≥ 0.10)")
        print("    These SHOULD separate the pair — worth investigating why the model ignores them:\n")
        for _, r in disc.iterrows():
            dominant = cwe_a if r["Delta_A_minus_B"] > 0 else cwe_b
            print(f"    [{r['Group']:>14}]  {r['Feature']:<28} "
                  f" {cwe_a}={r[f'Freq_{cwe_a}']:.2f}"
                  f"  {cwe_b}={r[f'Freq_{cwe_b}']:.2f}"
                  f"  Δ={r['Delta_A_minus_B']:+.3f}"
                  f"  ({dominant} dominates)")
    else:
        print("  ▶ No strongly discriminating features found (|Δ| < 0.10 for all)")

    print()
    amb = info["ambiguous"]
    if not amb.empty:
        print("  ▶ Ambiguous features (|Δ| < 0.05) — likely confusion triggers:\n")
        for _, r in amb.iterrows():
            print(f"    [{r['Group']:>14}]  {r['Feature']:<28} "
                  f" {cwe_a}={r[f'Freq_{cwe_a}']:.2f}"
                  f"  {cwe_b}={r[f'Freq_{cwe_b}']:.2f}")
    print()

────────────────────────────────────────────────────────────────────────────
  Pair: CWE195  ↔  CWE197
  Combined misclassifications : 10  (CWE195→CWE197: 10,  CWE197→CWE195: 0)
  Error rates                 : CWE195→CWE197=1.35%,  CWE197→CWE195=0.00%
  Dominant direction          : CWE195→CWE197 (10 times)
────────────────────────────────────────────────────────────────────────────
  ▶ Discriminating features (|Δ| ≥ 0.10)
    These SHOULD separate the pair — worth investigating why the model ignores them:

    [        memory]  pointer_arithmetic            CWE195=0.61  CWE197=0.00  Δ=+0.606  (CWE195 dominates)
    [        memory]  memset_present                CWE195=0.68  CWE197=0.21  Δ=+0.476  (CWE195 dominates)
    [        memory]  stack_buffer                  CWE195=0.45  CWE197=0.00  Δ=+0.455  (CWE195 dominates)
    [       integer]  cast_to_char                  CWE195=0.00  CWE197=0.40  Δ=-0.404  (CWE197 dominates)
    [       integer]  cast_to_short                 CWE195=

## 8. Global Feature Discriminability Ranking

In [9]:
global_rank = (
    master_df
    .groupby(["Feature", "Group"])["Abs_Delta"]
    .mean()
    .reset_index()
    .rename(columns={"Abs_Delta": "Mean_Abs_Delta"})
    .sort_values("Mean_Abs_Delta", ascending=False)
    .reset_index(drop=True)
)

print("Most discriminating features (highest avg |Δ| across all significant pairs):")
print(global_rank.head(15).to_string(index=False))
print()
print("Most ambiguous features (lowest avg |Δ| — hardest for model to use):")
print(global_rank.tail(10).sort_values("Mean_Abs_Delta").to_string(index=False))

Most discriminating features (highest avg |Δ| across all significant pairs):
             Feature          Group  Mean_Abs_Delta
  pointer_arithmetic         memory        0.423967
      memset_present         memory        0.330700
        stack_buffer         memory        0.312433
        cast_to_char        integer        0.275700
array_index_variable         memory        0.218900
  null_check_present error_handling        0.207433
       format_spec_s  format_string        0.187967
       cast_to_short        integer        0.153367
         uses_malloc         memory        0.101000
           uses_free         memory        0.101000
        strncpy_safe     string_ops        0.101000
     memmove_present         memory        0.101000
      memcpy_present         memory        0.101000
         sizeof_used         memory        0.085867
    INT_MAX_MIN_used        integer        0.068833

Most ambiguous features (lowest avg |Δ| — hardest for model to use):
         Feature     

## 9. Export

In [10]:
# Significant pairs selected from CM
pairs_df.to_csv("cwe_significant_pairs.csv", index=False)

# Full (pair × feature) master table
master_df.to_csv("cwe_pattern_analysis_master.csv", index=False)

# Per-CWE feature frequency matrix
cwe_freq.to_csv("cwe_feature_frequencies.csv")

# Global feature discriminability ranking
global_rank.to_csv("cwe_feature_discriminability.csv", index=False)

# Top-20 features per pair (sorted by |Δ|)
top_per_pair = pd.concat([
    info["pair_df"]
    .head(20)
    .assign(CWE_A=cwe_a, CWE_B=cwe_b,
            Combined_Count=info["combined_count"],
            Max_Rate=info["max_rate"])
    for (cwe_a, cwe_b), info in pair_summaries.items()
])
top_per_pair.to_csv("cwe_top_features_per_pair.csv", index=False)

print("✅ Exported:")
print("   cwe_significant_pairs.csv         — pairs selected from CM with both thresholds")
print("   cwe_pattern_analysis_master.csv   — full (pair × feature) table")
print("   cwe_feature_frequencies.csv       — per-CWE feature frequency matrix")
print("   cwe_feature_discriminability.csv  — global feature ranking by avg |Δ|")
print("   cwe_top_features_per_pair.csv     — top-20 features per significant pair")

✅ Exported:
   cwe_significant_pairs.csv         — pairs selected from CM with both thresholds
   cwe_pattern_analysis_master.csv   — full (pair × feature) table
   cwe_feature_frequencies.csv       — per-CWE feature frequency matrix
   cwe_feature_discriminability.csv  — global feature ranking by avg |Δ|
   cwe_top_features_per_pair.csv     — top-20 features per significant pair
